In [4]:
import sys
import os

# 1. Direct Python to the root project folder
sys.path.append(os.path.abspath(os.path.join('..')))

In [2]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
vs_index = VectorSearchIndex(
    mode="ivf",
    db_path="../data/faq_vectors2.db"
)

In [6]:
from scripts.rag_helper import RAGBase

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        
        return self.index.search(
            query_vector,
            num_results=num_results
        )


In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI()

In [8]:
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client
)

In [9]:
query = 'How to signup for an account?'
vector_assistant.rag(query)

'To create an account, click on the **“Sign Up”** button on the **top right corner** of our website and follow the instructions to complete the registration process.'